In [51]:
import numpy as np
import simpy
import pandas as pd
import itertools
import math
import matplotlib.pyplot as plt


In [52]:
class Exponential:
    '''
    Convenience class for the exponential distribution.
    packages up distribution parameters, seed and random generator.
    '''
    def __init__(self, mean, random_seed=None):
        '''
        Constructor
        
        Params:
        ------
        mean: float
            The mean of the exponential distribution
        
        random_seed: int, optional (default=None)
            A random seed to reproduce samples.  If set to none then a unique
            sample is created.
        '''
        self.rand = np.random.default_rng(seed=random_seed)
        self.mean = mean
        
    def sample(self, size=None):
        '''
        Generate a sample from the exponential distribution
        
        Params:
        -------
        size: int, optional (default=None)
            the number of samples to return.  If size=None then a single
            sample is returned.
        '''
        return self.rand.exponential(self.mean, size=size)



class Lognormal:
    """
    Encapsulates a lognormal distirbution
    """
    def __init__(self, mean, stdev, random_seed=None):
        """
        Params:
        -------
        mean = mean of the lognormal distribution
        stdev = standard dev of the lognormal distribution
        """
        self.rand = np.random.default_rng(seed=random_seed)
        mu, sigma = self.normal_moments_from_lognormal(mean, stdev**2)
        self.mu = mu
        self.sigma = sigma
        
    def normal_moments_from_lognormal(self, m, v):
        '''
        Returns mu and sigma of normal distribution
        underlying a lognormal with mean m and variance v
        source: https://blogs.sas.com/content/iml/2014/06/04/simulate-lognormal
        -data-with-specified-mean-and-variance.html

        Params:
        -------
        m = mean of lognormal distribution
        v = variance of lognormal distribution
                
        Returns:
        -------
        (float, float)
        '''
        phi = math.sqrt(v + m**2)
        mu = math.log(m**2/phi)
        sigma = math.sqrt(math.log(phi**2/m**2))
        return mu, sigma
        
    def sample(self):
        """
        Sample from the normal distribution
        """
        return self.rand.lognormal(self.mu, self.sigma)


class Normal:
    
    def __init__(self, mean, std, random_seed=None):
        self.rand = np.random.default_rng(seed=random_seed)
        self.mean = mean
        self.std = std
        
    def sample(self, size=None):
        sample = self.rand.normal(loc=self.mean, scale=self.std, size=size)
        while sample < 0:
           sample = self.rand.normal(loc=self.mean, scale=self.std, size=size)
        return sample

In [53]:
def trace(msg):
    '''
    Utility function for printing simulation
    set the TRACE constant to FALSE to 
    turn tracing off.
    
    Params:
    -------
    msg: str
        string to print to screen.
    '''
    if TRACE:
        print(msg)

In [54]:
# These are the parameters for the model

# resource counts
N_BED = 27


# time between arrivals in hours (exponential)
Emergency_IAT = 37.03
Ward_IAT = 25.97
Other_IAT = 47.1
AnE_IAT = 22.72
Xray_IAT = 575.09

# mean inter-arrival time in hours (Normal distribution)
Elective_mean_IAT = 19.51
# standard deviation of arrival times
Elective_std_IAT = 26.42

# mean Length of stay (Lognormal)
mean_Emergency_LOS = 140.15
mean_Ward_LOS = 177.89
mean_Other_LOS = 212.86
mean_AnE_LOS = 128.86
mean_Xray_LOS = 87.53
mean_Elective_LOS = 57.34


# Standard deviation of length of stay (Lognormal)
std_Emergency_LOS = 218.02
std_Ward_LOS = 276.54
std_Other_LOS = 457.67
std_AnE_LOS = 267.51
std_Xray_LOS = 108.15
std_Elective_LOS = 99.78


# SEEDS to reproduce results of a single run
REPRODUCIBLE_RUN = True
    
if REPRODUCIBLE_RUN:
    SEEDS = [42, 101, 1066, 1966, 2013, 999, 1444, 2016, 1456, 56,78,456]
else:
    SEEDS = [None, None, None, None, None, None, None, None]

In [59]:

class Scenario:
    '''
    Parameter container class for the Critical Care Unit (CCU) model.

    This class encapsulates all resource capacities, arrival rates, and 
    probability distribution parameters for a single simulation run. It utilizes 
    the provided global constants as defaults but allows for granular 
    parameterization of experimental scenarios via the constructor.

    Attributes
    ----------
    name : str or None
        A descriptive name for the scenario.
    N_bed : int
        The number of available triage bay resources.
    arrival_dist_emerg : Exponential
        Distribution for inter-arrival times from emergency
    arrival_dist_ward :  Exponential
        Distribution for inter-arrival times from the ward
    arrival_dist_other : Exponential
        Distribution for inter-arrival times from other sources
    arrival_dist_AnE : Exponential
        Distribution for inter-arrival times from A/E
    arrival_dist_xray : Exponential
        Distribution for inter-arrival times from Xray
    arrival_dist_elective : Normal Distribution
        Distribution for inter-arrival times.
    emerg_dist : Lognormal
        Distribution for the length of stay for emergency patients
    ward_dist : Lognormal
        Distribution for the length of stay for ward patients
    other_dist : Lognormal
        Distribution for the length of stay for ward patients
    AnE_dist : Lognormal
        Distribution for the length of stay for ward patients
    xray_dist : Lognormal
        Distribution for the length of stay for ward patients
    elect_dist : Lognormal
        Distribution for the length of stay for ward patients
    
    '''
    def __init__(
        self, 
        name=None,
        # Resources
        CCU_beds = N_BED,
        # Arrival parameters for Unplanned patients
        emerg_iat = Emergency_IAT,
        ward_iat = Ward_IAT,
        other_iat = Other_IAT,
        AnE_iat = AnE_IAT,
        xray_iat = Xray_IAT ,
        # Arrival parameters for Unplanned patients
        elective_mean_iat = Elective_mean_IAT,
        elective_std_iat = Elective_std_IAT,
        # LOS parameters for unplanned patients
        # Emergency
          emerg_mean_los = mean_Emergency_LOS,
          emerg_std_los = std_Emergency_LOS,
        # Ward 
           ward_mean_los = mean_Ward_LOS ,
           ward_std_los = std_Ward_LOS,
        # Other
           other_mean_los = mean_Other_LOS,
           other_std_los = std_Other_LOS,
        # Xray
           xray_mean_los = mean_Xray_LOS,
           xray_std_los = std_Xray_LOS,
        # AnE
           AnE_mean_los = mean_AnE_LOS,
           AnE_std_los = std_AnE_LOS,
        # LOS parameters for unplanned patients
        # Elective
           elect_mean_los = mean_Elective_LOS,
           elect_std_los = std_Elective_LOS,
        # Random Seeds
        seeds=SEEDS
    ):
        '''
        The init method sets up our defaults using the global constants.
        Any parameter can be overridden by passing a value to the constructor.
        
        Params:
        -------
        name : str or None
            optional name for scenario
        '''
        # optional name
        self.name = name
        
        # triage bays
        self.CCU_beds = N_BED
        
        # inter-arrival distribution for unplanned cases
        self.arrival_dist_emerg = Exponential( emerg_iat, random_seed=seeds[0])
        self.arrival_dist_ward = Exponential( ward_iat, random_seed=seeds[1])
        self.arrival_dist_other = Exponential(other_iat, random_seed=seeds[2])
        self.arrival_dist_AnE = Exponential( AnE_iat, random_seed=seeds[3])
        self.arrival_dist_xray = Exponential( xray_iat , random_seed=seeds[4])

       # inter-arrival distribution for planned cases
        self.arrival_dist_elective = Normal(elective_mean_iat,elective_std_iat, random_seed=seeds[5])

       #length of stay distribution for unplanned patients
        self.emerg_dist = Lognormal(emerg_mean_los, emerg_std_los, 
                                         random_seed=seeds[6])
        self.ward_dist = Lognormal(ward_mean_los, ward_std_los, 
                                         random_seed=seeds[7])
        self.other_dist = Lognormal(other_mean_los, other_std_los, 
                                         random_seed=seeds[8])
        self.AnE_dist = Lognormal(AnE_mean_los, AnE_std_los, 
                                         random_seed=seeds[9])
        self.xray_dist = Lognormal(xray_mean_los, xray_std_los, 
                                         random_seed=seeds[10])
        self.elect_dist = Lognormal(elect_mean_los, elect_std_los, 
                                         random_seed=seeds[11])


In [56]:
class CCU_Patients:
    """
    Simulates one CCU patient.
    """

    def __init__(self, identifier, env, args, group):
        self.identifier = identifier
        self.env = env
        self.args = args
        self.group = group

        self.CCU_beds = args.CCU_beds

        self.wait_time = 0
        self.planned_cancellations = 0
        self.total_bed_busy_time = 0
        self.total_patients_admitted = 0

    def assessment(self):
        """
        Patient process:
        - unplanned: request bed, wait if needed, stay, discharge, clean bed
        - planned: if no bed available at arrival, cancel; otherwise admit, stay, discharge, clean bed
        """
        arrival_time = self.env.now


        if self.group["type"] == "planned":
           if self.CCU_beds.count >= self.CCU_beds.capacity:
              self.planned_cancellations += 1
              trace(f'{self.group["name"]}-patient {self.identifier} PLANNED CANCELLED at {self.env.now:.2f}')
              return
     
              trace(f'{self.group["name"]}-patient {self.identifier} PLANNED ARRIVED at {self.env.now:.2f}')

        elif self.group["type"] == "unplanned":
           trace(f'{self.group["name"]}-patient {self.identifier} ARRIVED at {self.env.now:.2f}')

        with self.CCU_beds.request(priority=self.group["priority"]) as req:
           yield req

            # waiting time 
           self.wait_time = self.env.now - arrival_time
           

           trace(f'{self.group["name"]}-patient {self.identifier} ENTERED CCU after waiting {self.wait_time:.2f}')
      
           los = self.group["los_dist"].sample()

            # occupancy tracking
           self.total_bed_busy_time += los
           self.total_patients_admitted += 1

           yield self.env.timeout(los)

           trace(f'{self.group["name"]}-patient {self.identifier} DISCHARGED AT {self.env.now:.2f}')

           cleaning_time = 5
           self.total_bed_busy_time += cleaning_time
           yield self.env.timeout(cleaning_time)

           trace(f'bed ready after cleaning at {self.env.now:.2f}')

In [57]:
class CCU_model:
    
    def __init__(self, env, args):
        self.env = env
        self.args = args 
        self.init_model_resources(args)
        self.patients = []


    def init_model_resources(self, args):
        '''
        Setup the simpy resource objects
        
        Params:
        ------
        args - Scenario
            Simulation Parameter Container
        '''
        args.CCU_beds = simpy.PriorityResource(self.env, 
                                          capacity=args.CCU_beds)

    def arrival_dist(self, args):
        
        self.group = [
         {
            "name": "emergency",
            "arrival_dist": args.arrival_dist_emerg,
            "los_dist": args.emerg_dist,
            "type": "unplanned",
            "priority": 0
        },
         {
            "name": "ward",
            "arrival_dist": args.arrival_dist_ward,
            "los_dist": args.ward_dist,
            "type": "unplanned",
            "priority": 0
        },
            {
            "name": "other",
            "arrival_dist": args.arrival_dist_other,
            "los_dist": args.other_dist,
            "type": "unplanned",
            "priority": 0
        },
            {
            "name": "xray",
            "arrival_dist": args.arrival_dist_xray,
            "los_dist": args.xray_dist,
            "type": "unplanned",
            "priority": 0
        },
            {
            "name": "AnE",
            "arrival_dist": args.arrival_dist_AnE,
            "los_dist": args.AnE_dist,
            "type": "unplanned",
            "priority": 0
        },
            {
            "name": "elective",
            "arrival_dist": args.arrival_dist_elective,
            "los_dist": args.elect_dist,
            "type": "planned",
            "priority": 1
        }
        ]


    def arrivals_generator(self, group):
        '''
        IAT is exponentially distributed

        Parameters:
        ------
        env: simpy.Environment

        args: Scenario
            Container class for model data inputs
        '''

        for patient_count in itertools.count(start=1):
            inter_arrival_time = group['arrival_dist'].sample()
            yield self.env.timeout(inter_arrival_time)

        # create a new minor patient and pass in env and args
            new_patient = CCU_Patients(patient_count, self.env, self.args, group)

            #trace(f'patient {patient_count} arrives at: {self.env.now:.3f}')
            
            # keep a record of the patient for results calculation
            self.patients.append(new_patient)
            
            # init the CCU process for this patient
            self.env.process(new_patient.assessment())

In [58]:
# run length in hours
RUN_LENGTH = 365*24

# Turn off tracing
TRACE = False

# create simpy environment
env = simpy.Environment()

# base case scenario with default parameters
default_args = Scenario()

# create the model
model = CCU_model(env, default_args)
model.arrival_dist(default_args)  

# start a generator for each group
for group in model.group:
    env.process(model.arrivals_generator(group))
    

env.run(until=RUN_LENGTH)
print(f'end of run. simulation clock time = {env.now}')

mean_waiting_time_unplanned = np.array([patient.wait_time
                                for patient in model.patients]).mean()

total_patients_admitted = np.array([patient.total_patients_admitted
                                for patient in model.patients]).sum()

total_planned_cancellations = np.array([patient.planned_cancellations
                               for patient in model.patients]).sum()
bed_occupancy = np.array([patient.total_bed_busy_time for patient in model.patients])/ (24 * env.now)
bed_occupancy_rate = np.array(bed_occupancy).sum()

print('\nSingle run results\n------------------')
print(f'Total_patients_admitted: {total_patients_admitted:.2f}')
print(f'Total_planned_cancellations: {total_planned_cancellations:.2f}')
print(f'Mean_waiting_time_unplanned: {mean_waiting_time_unplanned:.2f}')
print(f'Bed occupancy rate (%): {bed_occupancy_rate * 100:.2f}%')

end of run. simulation clock time = 8760

Single run results
------------------
Total_patients_admitted: 1373.00
Total_planned_cancellations: 64.00
Mean_waiting_time_unplanned: 5.57
Bed occupancy rate (%): 92.76%
